## Day 3: Baselines and Linear Regression

This notebook implements:
1) baseline volatility forecasts,
2) linear regression forecasting,
3) MAE/RMSE evaluation,
4) prediction-vs-realized plots for AAPL and MSFT.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error


PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name.startswith("notebooks") else Path.cwd().resolve()
DATA_DIR = PROJECT_ROOT / "data:"
if not DATA_DIR.exists():
    DATA_DIR = PROJECT_ROOT / "data"

FIG_DIR = PROJECT_ROOT / "figures:"
if not FIG_DIR.exists():
    FIG_DIR = PROJECT_ROOT / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print("Using data dir:", DATA_DIR)
print("Using figures dir:", FIG_DIR)

Using data dir: /Users/syedm/Desktop/MSAI/CSML/CSML FInal Project/data:
Using figures dir: /Users/syedm/Desktop/MSAI/CSML/CSML FInal Project/figures:


In [2]:
def add_volatility(df, window=10):
    df = df.sort_values("Date").copy()
    df["log_return"] = np.log(df["Close"]).diff()
    df["rv_10"] = df["log_return"].rolling(window).std()
    return df


def make_target(df):
    df = df.sort_values("Date").copy()
    df["target_rv_10"] = df["rv_10"].shift(-1)
    return df


def add_features(df, n_return_lags=5):
    df = df.sort_values("Date").copy()

    df["log_close"] = np.log(df["Close"])

    for k in range(1, n_return_lags + 1):
        df[f"log_return_lag_{k}"] = df["log_return"].shift(k)

    df["rv_10_lag_1"] = df["rv_10"].shift(1)
    df["log_volume"] = np.log(df["Volume"].replace(0, np.nan))
    df["volume_change"] = df["Volume"].pct_change()

    return df


def build_day2_ready_df(path):
    df = pd.read_csv(path, parse_dates=["Date"])
    # Convert to tz-naive timestamps for consistent time slicing.
    df["Date"] = pd.to_datetime(df["Date"], utc=True).dt.tz_localize(None)

    df = make_target(add_volatility(df, window=10))
    df = add_features(df, n_return_lags=5)
    df = df.dropna().sort_values("Date").reset_index(drop=True)
    return df


aapl = build_day2_ready_df(DATA_DIR / "AAPL_clean.csv")
msft = build_day2_ready_df(DATA_DIR / "MSFT_clean.csv")

print("AAPL rows:", len(aapl), "MSFT rows:", len(msft))
aapl.head()

AAPL rows: 1244 MSFT rows: 1244


,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits,Company,log_return,...,target_rv_10,log_close,log_return_lag_1,log_return_lag_2,log_return_lag_3,log_return_lag_4,log_return_lag_5,rv_10_lag_1,log_volume,volume_change
0,2018-12-19 05:00:00,39.832146,40.180077,38.174072,38.605988,196189200,0.0,0.0,AAPL,-0.031688,...,0.019214,3.653407,0.012909,-0.009349,-0.032521,0.010881,0.002783,0.020648,19.094590,0.449324
1,2018-12-20 05:00:00,38.488408,38.898729,37.264651,37.631779,259092000,0.0,0.0,AAPL,-0.025559,...,0.019733,3.627849,-0.031688,0.012909,-0.009349,-0.032521,0.010881,0.018513,19.372694,0.320623
2,2018-12-21 05:00:00,37.638979,37.950918,35.904122,36.168068,382978400,0.0,0.0,AAPL,-0.039672,...,0.019180,3.588177,-0.025559,-0.031688,0.012909,-0.009349,-0.032521,0.019214,19.763489,0.478156
3,2018-12-24 05:00:00,35.548997,36.364839,35.174672,35.232262,148676800,0.0,0.0,AAPL,-0.026214,...,0.032476,3.561962,-0.039672,-0.025559,-0.031688,0.012909,-0.009349,0.019733,18.817285,-0.611788
4,2018-12-26 05:00:00,35.584984,37.727760,35.205859,37.713364,234330000,0.0,0.0,AAPL,0.068052,...,0.032296,3.630015,-0.026214,-0.039672,-0.025559,-0.031688,0.012909,0.019180,19.272241,0.576103


In [3]:
feature_cols = [
    c
    for c in aapl.columns
    if c.startswith("log_return_lag_")
    or c in ["log_close", "rv_10", "rv_10_lag_1", "log_volume", "volume_change"]
]
target_col = "target_rv_10"


def time_split(df, train_end, val_end):
    train = df[df["Date"] <= train_end].copy()
    val = df[(df["Date"] > train_end) & (df["Date"] <= val_end)].copy()
    test = df[df["Date"] > val_end].copy()
    return train, val, test


train_end = pd.Timestamp("2021-12-31")
val_end = pd.Timestamp("2022-12-31")

a_train, a_val, a_test = time_split(aapl, train_end, val_end)
m_train, m_val, m_test = time_split(msft, train_end, val_end)

print("Feature columns:", feature_cols)
print("AAPL split sizes:", len(a_train), len(a_val), len(a_test))
print("MSFT split sizes:", len(m_train), len(m_val), len(m_test))

Feature columns: ['rv_10', 'log_close', 'log_return_lag_1', 'log_return_lag_2', 'log_return_lag_3', 'log_return_lag_4', 'log_return_lag_5', 'rv_10_lag_1', 'log_volume', 'volume_change']
AAPL split sizes: 764 252 228
MSFT split sizes: 764 252 228


In [4]:
def naive_baseline(df):
    df = df.sort_values("Date").copy()
    # target_rv_10 at row t is rv_10 at row t+1, so current rv_10 is a valid naive forecast.
    df["pred_naive"] = df["rv_10"]
    return df


def rolling_mean_baseline(df, window=10):
    df = df.sort_values("Date").copy()
    df["pred_rollmean"] = df["rv_10"].rolling(window).mean()
    return df


def add_baselines(train, val, test):
    combined = pd.concat([train, val, test], axis=0).sort_values("Date").copy()
    combined = naive_baseline(combined)
    combined = rolling_mean_baseline(combined, window=10)
    combined = combined.dropna(subset=["pred_naive", "pred_rollmean", target_col])
    return time_split(combined, train_end, val_end)


def eval_forecasts(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return mae, rmse


def evaluate_baselines(df_test, ticker):
    y_true = df_test[target_col].values
    mae_naive, rmse_naive = eval_forecasts(y_true, df_test["pred_naive"])
    mae_rm, rmse_rm = eval_forecasts(y_true, df_test["pred_rollmean"])
    return [
        {"Ticker": ticker, "Model": "Naive", "Split": "Test", "MAE": mae_naive, "RMSE": rmse_naive},
        {"Ticker": ticker, "Model": "RollingMean10", "Split": "Test", "MAE": mae_rm, "RMSE": rmse_rm},
    ]


a_train, a_val, a_test = add_baselines(a_train, a_val, a_test)
m_train, m_val, m_test = add_baselines(m_train, m_val, m_test)

baseline_rows = []
baseline_rows.extend(evaluate_baselines(a_test, "AAPL"))
baseline_rows.extend(evaluate_baselines(m_test, "MSFT"))

baseline_results = pd.DataFrame(baseline_rows)
baseline_results

,Ticker,Model,Split,MAE,RMSE
0,AAPL,Naive,Test,0.000956,0.001670
1,AAPL,RollingMean10,Test,0.002485,0.003306
2,MSFT,Naive,Test,0.001195,0.002069
3,MSFT,RollingMean10,Test,0.003382,0.004327


In [5]:
def fit_linear(train, val, test, label):
    X_train = train[feature_cols].values
    y_train = train[target_col].values
    X_val = val[feature_cols].values
    y_val = val[target_col].values
    X_test = test[feature_cols].values
    y_test = test[target_col].values

    model = LinearRegression()
    model.fit(X_train, y_train)

    y_val_pred = model.predict(X_val)
    y_test_pred = model.predict(X_test)

    mae_val, rmse_val = eval_forecasts(y_val, y_val_pred)
    mae_test, rmse_test = eval_forecasts(y_test, y_test_pred)

    test_out = test.copy()
    test_out["pred_linear"] = y_test_pred

    metrics_rows = [
        {"Ticker": label, "Model": "LinearRegression", "Split": "Val", "MAE": mae_val, "RMSE": rmse_val},
        {"Ticker": label, "Model": "LinearRegression", "Split": "Test", "MAE": mae_test, "RMSE": rmse_test},
    ]
    return model, test_out, metrics_rows


a_lr, a_test, a_lr_rows = fit_linear(a_train, a_val, a_test, "AAPL")
m_lr, m_test, m_lr_rows = fit_linear(m_train, m_val, m_test, "MSFT")

linear_results = pd.DataFrame(a_lr_rows + m_lr_rows)
linear_results

,Ticker,Model,Split,MAE,RMSE
0,AAPL,LinearRegression,Val,0.001722,0.002639
1,AAPL,LinearRegression,Test,0.001231,0.001792
2,MSFT,LinearRegression,Val,0.001982,0.003048
3,MSFT,LinearRegression,Test,0.001310,0.002024


In [6]:
results = pd.concat([baseline_results, linear_results], ignore_index=True)
results = results.sort_values(["Ticker", "Split", "RMSE"]).reset_index(drop=True)

results

,Ticker,Model,Split,MAE,RMSE
0,AAPL,Naive,Test,0.000956,0.001670
1,AAPL,LinearRegression,Test,0.001231,0.001792
2,AAPL,RollingMean10,Test,0.002485,0.003306
3,AAPL,LinearRegression,Val,0.001722,0.002639
4,MSFT,LinearRegression,Test,0.001310,0.002024
5,MSFT,Naive,Test,0.001195,0.002069
6,MSFT,RollingMean10,Test,0.003382,0.004327
7,MSFT,LinearRegression,Val,0.001982,0.003048


In [7]:
RESULTS_DIR = DATA_DIR / "results_day3"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

results.to_csv(RESULTS_DIR / "metrics_day3.csv", index=False)
a_test.to_csv(RESULTS_DIR / "AAPL_test_with_predictions.csv", index=False)
m_test.to_csv(RESULTS_DIR / "MSFT_test_with_predictions.csv", index=False)

print("Saved result files to:", RESULTS_DIR)
for p in sorted(RESULTS_DIR.glob("*.csv")):
    print(" -", p.name)

Saved result files to: /Users/syedm/Desktop/MSAI/CSML/CSML FInal Project/data:/results_day3
 - AAPL_test_with_predictions.csv
 - MSFT_test_with_predictions.csv
 - metrics_day3.csv


In [8]:
def plot_preds(df_test, ticker):
    plt.figure(figsize=(10, 4))
    plt.plot(df_test["Date"], df_test[target_col], label="Realized RV", linewidth=1.8)
    plt.plot(df_test["Date"], df_test["pred_naive"], label="Naive", alpha=0.8)
    plt.plot(df_test["Date"], df_test["pred_linear"], label="Linear", alpha=0.8)
    plt.title(f"{ticker} - 1-day-ahead 10-day Realized Volatility")
    plt.xlabel("Date")
    plt.ylabel("Volatility")
    plt.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / f"{ticker}_baseline_vs_linear.png", dpi=150)
    plt.close()


plot_preds(a_test, "AAPL")
plot_preds(m_test, "MSFT")

print("Saved plots:")
print(" -", (FIG_DIR / "AAPL_baseline_vs_linear.png").name)
print(" -", (FIG_DIR / "MSFT_baseline_vs_linear.png").name)

Saved plots:
 - AAPL_baseline_vs_linear.png
 - MSFT_baseline_vs_linear.png


### Day 3 Checklist (Implemented)

- Baselines: naive and rolling-mean forecasts implemented and evaluated.
- ML model: linear regression trained on train, validated on val, tested on test.
- Metrics: MAE and RMSE table created and saved.
- Figures: baseline-vs-linear prediction plots saved for AAPL and MSFT.